# 7. LangChain Agent 활용

## 학습 목표

1. LLM의 한계와 Agent가 필요한 이유를 설명할 수 있습니다.
2. ReAct(Reasoning + Acting) 동작 원리를 이해합니다.
3. `@tool`로 도구를 만들고, `bind_tools`로 도구 호출 과정을 직접 확인합니다.
4. `create_agent`로 검색·계산·이미지 생성 도구를 쓰는 Agent를 만듭니다.

**기준 버전**: LangChain 1.x, LangGraph 1.x, 모델 `gpt-6-luna`

> **버전 변경 사항**
> - Agent 생성 함수는 `langchain.agents.create_agent`로 통일되었습니다.
> - `langgraph.prebuilt.create_react_agent`는 deprecated입니다.
> - `AgentExecutor`(`langchain_classic`)는 레거시입니다. 반복 실행 관리는 `create_agent` 내부에서 처리합니다.

## 1. LLM의 결정적인 한계

1. **최신 정보의 부재**: 학습이 끝난 이후의 실시간 정보(오늘 날씨, 현재 주가 등)를 알지 못합니다.
2. **계산 취약성**: 다음 단어를 예측하는 구조라 복잡한 연산에서 실수할 수 있습니다.

### 해결책: LLM에게 "도구(Tools)"를 쥐여 주자!

사람은 계산이 어려우면 계산기를 쓰고, 모르는 것은 검색합니다. AI에게도 검색, 계산기, 데이터베이스 같은 도구를 주고 상황에 맞게 스스로 쓰게 만들 수 있습니다.

> **Agent(에이전트)의 정의**
> LLM을 두뇌로 삼아, 사용자의 요청을 해결하기 위해 **어떤 도구를, 어떤 순서로 사용할지 스스로 판단하고 실행**하는 AI 시스템입니다.

## 2. 핵심 구성 요소

| 구성 요소 | 역할 |
|---|---|
| Model (LLM) | 생각하고 다음 행동을 결정하는 두뇌 |
| Tools | 검색, 계산 등 실행 가능한 도구 |
| Agent (`create_agent`) | 모델과 도구를 묶어 "생각 → 도구 실행 → 결과 확인" 루프를 반복하는 실행 엔진 |

## 3. Agent의 핵심 동작 원리: ReAct (Reasoning + Acting)

```text
사용자 입력 → [Thought → Action → Tool 실행 → Observation] × N → Final Answer
```

1. **Thought(생각)**: 지금 무엇을 해야 할지 판단합니다. 예: "서울 날씨를 알아야 하니 날씨 도구를 쓰자."
2. **Action(행동)**: 사용할 도구를 선택합니다. 예: `get_weather`
3. **Action Input(행동 입력)**: 도구에 넣을 값을 정합니다. 예: `"Seoul"`
4. **Observation(관찰)**: 도구가 돌려준 결과를 확인합니다. 예: "맑음, 15°C"
5. **반복**: 정보가 충분하면 최종 답변을 만들고, 아니면 1~4를 반복합니다.

## 4. Agent의 강점

- **자동 도구 선택**: 질문에 따라 검색, 계산기, API 등을 스스로 선택합니다.
- **다단계 추론**: "검색 → 정보 추출 → 계산 → 결론" 과정을 자동화합니다.
- **실시간 정보**: 학습 범위를 넘어선 최신 정보를 처리합니다.

### Chat Model vs Agent

| 구분 | Chat Model (챗봇) | Agent |
|---|---|---|
| 비유 | 아무 도구 없이 시험을 보는 학생 | 오픈북 시험에서 컴퓨터와 계산기를 쓰는 학생 |
| 문제 해결 | 학습한 지식만 활용 | 필요하면 외부 도구 사용 |
| 최신 정보 | 학습 시점 이후 정보는 모름 | 검색 도구로 확인 가능 |
| 계산 능력 | 복잡한 계산은 실수 가능 | 계산기·코드 실행 도구 활용 |
| 외부 시스템 연동 | 불가능 | DB, API, 파일 등과 연동 |
| 흐름 | 질문 → 답변 | 질문 → 계획 → 도구 사용 → 답변 |

### Agent와 RAG의 차이

| 구분 | RAG | Agent |
|---|---|---|
| 목적 | 정적 문서 검색 → 답변 생성 | 동적 도구 선택 → 작업 수행 |
| 입력 | 고정된 문서/벡터DB | 다양한 도구(검색, 계산기, API 등) |
| 흐름 | 검색 → 컨텍스트 → LLM 한 번 호출 | 생각 → 도구 선택 → 실행 → 반복 |
| 복잡도 | 단순 (검색 + 생성) | 복합 (의사결정 + 실행) |
| 예시 | "매뉴얼에서 OS 버전 확인" | "날씨 검색 → 온도 변환 → 비교" |

```python
# RAG: 고정 흐름 (항상 검색 실행)
rag_chain = retriever | prompt | llm

# Agent: 동적 판단 ("검색이 필요한가?"를 스스로 판단)
agent = create_agent(llm, tools)
```

### 실습 환경 준비

터미널에서 필요한 패키지를 설치하고, 프로젝트 최상위 폴더의 `.env` 파일에 API 키와 모델명을 저장합니다.

```text
OPENAI_API_KEY="YOUR_OPENAI_API_KEY"
OPENAI_DEFAULT_MODEL="gpt-6-luna"
```

> API 키는 코드 셀이나 GitHub에 직접 입력하지 마세요.

```bash
uv add langchain langchain-openai langgraph langchain-tavily wikipedia python-dotenv
```

웹 검색 실습(8장)에는 [Tavily](https://tavily.com) API 키가 필요합니다. `.env`에 추가합니다.

```text
TAVILY_API_KEY="YOUR_TAVILY_API_KEY"
```

In [ ]:
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI

load_dotenv(override=True)

default_model = os.getenv("OPENAI_DEFAULT_MODEL", "gpt-6-luna")
os.getenv("TAVILY_API_KEY")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env 파일에 OPENAI_API_KEY를 설정한 뒤 다시 실행하세요.")

# reasoning_effort="none": 수업 예제는 빠른 응답과 낮은 비용을 위해 추론 기능을 끕니다.
llm = ChatOpenAI(model=default_model, reasoning_effort="none")
print("사용 모델:", default_model)

> **gpt-6-luna 사용 시 참고**
> - `reasoning_effort`는 `none` / `low` / `medium`(기본) / `high` / `xhigh` / `max`를 지원합니다. 강의 예제는 `none`으로 통일합니다.
> - `temperature`는 추론이 꺼져 있을 때(`reasoning_effort="none"`)만 적용됩니다.
> - 모델명·요금·지원 파라미터는 바뀔 수 있으므로 수업 전에 [OpenAI 모델 문서](https://developers.openai.com/api/docs/models/gpt-6-luna)를 확인하세요.

> `gpt-6` 계열 모델에 도구를 연결하면 `langchain-openai`가 자동으로 Responses API로 요청합니다. 별도 설정은 필요 없습니다.

## 5. 도구(Tool) 정의하기

- Tool은 Agent가 사용할 수 있는 **외부 기능(함수)** 입니다.
- `@tool` 데코레이터를 붙이면 일반 파이썬 함수가 Tool로 등록됩니다.
- Agent는 **함수 이름, 타입 힌트, Docstring(설명)** 을 읽고 언제 이 도구를 쓸지 판단합니다. 그래서 Docstring을 구체적으로 써야 합니다.

In [ ]:
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """두 숫자를 더합니다."""
    return a + b

@tool
def get_word_length(word: str) -> int:
    """단어나 문장의 정확한 글자 수(길이)를 계산할 때 사용합니다."""
    return len(word)

print(add.name, "-", add.description)
print(add.args)
print(add.invoke({"a": 5, "b": 7}))

## 6. 도구 호출 과정 직접 보기 - `bind_tools`

Agent를 쓰기 전에, 모델이 도구를 **어떻게 요청하는지** 직접 확인합니다.

1. `llm.bind_tools(tools)`: 모델에게 사용할 수 있는 도구 목록을 알려 줍니다.
2. 모델은 답변 대신 **도구 호출 요청(`tool_calls`)** 을 반환합니다. (모델이 함수를 직접 실행하지는 않습니다.)
3. 우리 코드가 도구를 실행하고, 결과를 `ToolMessage`로 다시 모델에 전달합니다.
4. 모델이 도구 결과를 바탕으로 최종 답변을 만듭니다.

In [ ]:
from langchain_core.messages import ToolMessage

tools = [add, get_word_length]
tools_dict = {t.name: t for t in tools}

llm_with_tools = llm.bind_tools(tools)

query = "5와 7을 더해 줘"

# 1차 호출: 모델이 도구 호출을 요청
res1 = llm_with_tools.invoke(query)
print("Tool 호출 정보:", res1.tool_calls)

# 도구 실행
tool_call = res1.tool_calls[0]
result = tools_dict[tool_call["name"]].invoke(tool_call["args"])
print("Tool 실행 결과:", result)

# 2차 호출: 도구 결과를 모델에 전달 → 최종 답변
messages = [
    ("human", query),
    res1,
    ToolMessage(tool_call_id=tool_call["id"], content=str(result)),
]
res2 = llm_with_tools.invoke(messages)
print("최종 답변:", res2.text)   # .text: 응답 텍스트만 문자열로 꺼냄 (7장 참고)

### 6-1. 여러 번 도구를 써야 하는 경우 - 반복문으로 직접 구현

질문: "'만나서 반갑습니다'는 글자 수가 몇 개인지 알려 주고, 그 결과에 5를 더해 줘"

→ `get_word_length` 실행 후 그 결과로 `add`를 실행해야 합니다. 도구 호출이 없을 때까지 반복합니다.

In [ ]:
query = "'만나서 반갑습니다'는 글자 수가 몇 개인지 알려 주고, 그 결과에 5를 더해 줘"
messages = [("human", query)]

for _ in range(10):                       # 무한 반복 방지
    res = llm_with_tools.invoke(messages)
    messages.append(res)

    if not res.tool_calls:                # 도구 호출이 없으면 최종 답변
        break

    for call in res.tool_calls:
        result = tools_dict[call["name"]].invoke(call["args"])
        print(f"Tool '{call['name']}'({call['args']}) → {result}")
        messages.append(ToolMessage(tool_call_id=call["id"], content=str(result)))

print("\n최종 답변:", res.text)

## 7. create_agent - Agent 만들기

6장의 반복문(생각 → 도구 실행 → 결과 전달 → 반복)을 **`create_agent`가 대신 처리**합니다.

- 입력과 출력은 `{"messages": [...]}` 형태입니다. (LangGraph 기반이라 상태 키가 `messages`로 고정)
- 최종 답변은 `result["messages"][-1].text`로 꺼냅니다.

> **`.content` 대신 `.text`를 쓰는 이유**: `gpt-6` 계열 모델에 도구를 연결하면 `langchain-openai`가 자동으로 **Responses API**를 사용합니다. 이때 `.content`는 문자열이 아니라 블록 리스트(`[{"type": "text", "text": ...}]`)일 수 있습니다. `.text`는 어느 경우든 텍스트만 문자열로 돌려줍니다.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[add, get_word_length],
    system_prompt="당신은 필요할 때 도구를 사용하는 유능한 AI 비서입니다.",
)

query = "'만나서 반갑습니다'는 글자 수가 몇 개인지 알려 주고, 그 결과에 3을 더해 줘"
result = agent.invoke({"messages": [("human", query)]})

print(result["messages"][-1].text)

Agent 내부에서 오간 메시지를 보면 ReAct 과정을 확인할 수 있습니다.

In [ ]:
for msg in result["messages"]:
    if msg.type == "ai" and msg.tool_calls:
        print(f"[ai → 도구 요청] {[(c['name'], c['args']) for c in msg.tool_calls]}")
    else:
        print(f"[{msg.type}] {msg.text}")

> **반복 횟수 제한**: 예전 `AgentExecutor(max_iterations=4)` 대신, 실행할 때 `config={"recursion_limit": 10}`처럼 제한합니다.
> ```python
> agent.invoke({"messages": [("human", query)]}, config={"recursion_limit": 10})
> ```

In [ ]:
query = "'만나서 반갑습니다'는 글자 수가 몇 개인지 알려 주고, 그 결과에 3을 더해 줘"
result = agent.invoke({"messages": [("human", query)], "config": {"recursion_limit": 4}})

print(result["messages"][-1].text)

## 8. 웹 검색 Agent - Tavily

- 실시간 정보가 필요한 질문은 웹 검색 도구로 해결합니다.
- `TavilySearch`는 그 자체가 LangChain Tool이므로 `tools` 리스트에 바로 넣으면 됩니다.

In [ ]:
from langchain.agents import create_agent
from langchain_tavily import TavilySearch

web_search = TavilySearch(max_results=3)

agent = create_agent(
    model=llm,
    tools=[web_search, get_word_length, add],
    system_prompt="당신은 필요한 도구를 적절히 사용하여 질문에 답하는 유능한 AI 어시스턴트입니다.",
)

result = agent.invoke({
    "messages": [("human", "오늘 서울 날씨와 도쿄 날씨를 검색해서 비교해 줘")]
})
print(result["messages"][-1].text)

In [ ]:
result = agent.invoke({
    "messages": [("human", "현재 대한민국 대통령의 이름이 무엇인지 알아내고, 그 이름이 총 몇 글자(공백 제외)인지 계산해줘")]
})
print(result["messages"][-1].text)

In [ ]:
result = agent.invoke({
    "messages": [("human", "한글을 만든 조선시대 왕의 이름이 몇 글자인지 알려줘, 그 결과에 10을 더한 값도 알려줘")]
})
print(result["messages"][-1].text)

## 9. Wikipedia 검색 Agent

`wikipedia` 라이브러리로 검색 도구를 직접 만듭니다.

In [ ]:
import wikipedia
from langchain_core.tools import tool
from langchain.agents import create_agent

wikipedia.set_lang("ko")   # 한국어 위키백과 (영어는 "en")

@tool
def wiki_search(query: str) -> str:
    """위키백과에서 인물, 장소, 사건 등의 정보를 검색합니다."""
    try:
        title = wikipedia.search(query, results=1)[0]
        return wikipedia.summary(title, sentences=5)
    except Exception as e:
        return f"검색 실패: {e}"

agent = create_agent(
    model=llm,
    tools=[wiki_search],
    system_prompt="너는 필요할 때 도구를 사용하는 친절한 Agent야.",
)

result = agent.invoke({
    "messages": [("human", "에드 시런은 몇 년도에 태어났니? 2026년 기준으로 몇 살이니?")]
})
print(result["messages"][-1].text)

### 결과 구조 보기

Agent 결과의 `messages`에는 사용자 질문, 도구 요청, 도구 결과, 최종 답변이 순서대로 들어 있습니다.

In [ ]:
import json

json_result = {"messages": [{"type": m.type, "text": m.text} for m in result["messages"]]}
print(json.dumps(json_result, ensure_ascii=False, indent=2))

**동작 구조**

1. Agent가 질문을 받습니다. 예: "에드 시런은 언제 태어났나요? 몇 살인가요?"
2. 필요하면 도구를 사용합니다. 예: 위키백과에서 생년월일 검색
3. 중간 결과를 모아 필요하면 도구를 다시 호출합니다.
4. 모든 정보를 종합해 최종 답변을 만듭니다.

## 10. 실습 해보기 : 여러 도구를 쓰는 Agent - 계산기 + 날씨
    - "25 곱하기 4의 결과는 무엇인가요?",
    - "서울의 날씨는 어떤가요?",
    - "부산의 날씨를 알려 주고 100 나누기 4도 계산해 주세요.",

## 11. 이미지 생성 Agent

- 이미지 생성 API를 도구로 감싸 Agent에 연결합니다.
- DALL·E 3는 2026년 5월 12일 API에서 종료되었습니다. 이 예제는 OpenAI 이미지 모델 `gpt-image-2.5-flare`를 사용합니다.
- GPT Image 모델은 이미지를 **base64 데이터**로 반환하므로 파일로 저장합니다.
- 이미지 생성은 텍스트보다 비용이 크므로 실행 횟수에 주의하세요.

In [ ]:
import base64
from datetime import datetime
from openai import OpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent

client = OpenAI()

@tool
def generate_image(prompt: str) -> str:
    """이미지를 생성합니다. 입력: 생성할 이미지에 대한 상세한 설명. 출력: 저장된 이미지 파일 경로."""
    try:
        result = client.images.generate(
            model="gpt-image-2.5-flare",
            prompt=prompt,
            size="1024x1024",
        )
        filename = f"image_{datetime.now():%Y%m%d_%H%M%S}.png"
        with open(filename, "wb") as f:
            f.write(base64.b64decode(result.data[0].b64_json))
        return f"이미지가 생성되었습니다. 파일: {filename}"
    except Exception as e:
        return f"이미지 생성 중 오류 발생: {e}"

agent = create_agent(
    model=llm,
    tools=[generate_image],
    system_prompt="당신은 이미지 생성 요청을 받으면 도구를 사용해 이미지를 만들고 결과를 알려 주는 도우미입니다.",
)

result = agent.invoke({"messages": [("human", "고양이가 우주복을 입고 있는 귀여운 그림을 그려 줘")]})
print(result["messages"][-1].text)

In [ ]:
# 생성된 이미지 확인 (노트북에서 표시)
import glob
from IPython.display import Image, display

files = sorted(glob.glob("image_*.png"))
if files:
    display(Image(filename=files[-1], width=400))

## 12. LangChain에서 많이 쓰는 도구 종류

| 카테고리 | 예시 |
|---|---|
| 검색 | Tavily, Wikipedia, DuckDuckGo, Google/Bing 검색 |
| 계산·코드 실행 | 계산기, Python REPL, 코드 실행기 |
| 데이터베이스 | SQL, MongoDB, Redis 등 조회 도구 |
| 웹·API 호출 | 날씨, 뉴스 등 공공 API, HTTP 요청(GET/POST) |
| 멀티미디어 | 이미지 생성, 텍스트 음성 변환(TTS) |
| 파일·시스템 | 파일 읽기/쓰기, 쉘 명령 실행 |
| 외부 서비스 연결 | MCP 서버의 도구 (→ **9. MCP**) |

> 많은 외부 도구는 `langchain-tavily`처럼 **별도 통합 패키지**로 제공됩니다. 사용 전 해당 패키지의 최신 문서를 확인하세요.